<a href="https://colab.research.google.com/github/GimenesPaula/GimenesPaula/blob/main/Lanc_vendas_anos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bibliotecas Phyton

In [3]:
!pip install requests

In [4]:
pip install unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 4.2 MB/s eta 0:00:00


In [5]:
#Carrega Bibliotecas
import pandas as pd
import numpy as np
import re
from functools import lru_cache
from unidecode import unidecode
import requests

# Fazer Upload planilhas
1.   Lançamentos
2.   Inci
3.   Vendas

In [6]:
#Lançamentos
from google.colab import files
uploaded = files.upload()

Saving MIntel 2025 Br, Ar, Ch.xlsx to MIntel 2025 Br, Ar, Ch.xlsx


In [7]:
filename = next(iter(uploaded))

In [8]:
# INCI name produtos
from google.colab import files
inci = files.upload()

Saving inci_16_08.xlsx to inci_16_08.xlsx


In [9]:
filename2 = next(iter(inci))

In [10]:
#Vendas Distribuidores
from google.colab import files
dist = files.upload()

Saving vendas 2025_2024_2023.xlsx to vendas 2025_2024_2023.xlsx


In [49]:
filename4 = next(iter(dist))

# Análise Ferramenta de Vendas

In [50]:
data = '/content/'+filename4
df_sale = pd.read_excel(data)
pd.options.display.max_columns=None
df_sale.nunique()

,0
Distributor,6
Material,90
Customer,1995
2025,254
2024,350
2023,383
2022,361


### Gera chave para Fabricante

In [51]:
#Dicionários
PALAVRAS_IRRELEVANTES = {
    'industria', 'comercio', 'cosmetico', 'cosmeticos', 'tecnologia', 'ltda', 'me', 'eireli', 'sa', 'cia', 'comercial', 'ind', ' e ',
    'produtos', 'servicos', 'serviço', 'do', 'da', 'de', 'dos', 'das', 'the', 'group', 'grupo', 'laboratorio',
    'inc', 'corp', 'corporation', 'associados', 'associado', 'associacao', 'associação', 'holding', 'importadora',
    'exportadora', 'importacao', 'importação', 'exportacao', 'exportação', 'distribuidora', 'distribuidor', 'fabricacao',
    'fabricante', 'comerciante', 'comercio', 'comércio', 'comercial', 'empresa', 'sociedade', 'unipessoal',
    'aerosol', 'aerossol',
}

In [52]:
# Função para limpar texto
def limpar(texto):
    if pd.isnull(texto):
        return []
    texto = unidecode(str(texto)).lower()
    texto = re.sub(r'\b\d{2,}\b', '', texto)
    texto = re.sub(r'[.,/\\-]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return [p for p in texto.split(' ') if p]

# Função para extrair chave representativa
def extrair_chave(palavras):
    if not palavras:
        return ''
    relevantes = [p for p in palavras if p not in PALAVRAS_IRRELEVANTES]
    if not relevantes:
        relevantes = palavras
    if len(relevantes) >= 2:
        return f"{relevantes[0]} {relevantes[1]}"
    return relevantes[0]

# Função para gerar chaves para DataFrame com fallback para valor bruto da coluna1
def gerar_chaves(df, coluna1, coluna2=None, coluna3=None):
    palavras1 = df[coluna1].apply(limpar)
    palavras2 = df[coluna2].apply(limpar) if coluna2 else pd.Series([[]] * len(df))
    palavras3 = df[coluna3].apply(limpar) if coluna3 else pd.Series([[]] * len(df))

    def chave_final(idx):
        for lista in [palavras1.iloc[idx], palavras2.iloc[idx], palavras3.iloc[idx]]:
            chave = extrair_chave(lista)
            if chave:
                return chave
        return str(df[coluna1].iloc[idx]).strip().lower()  # fallback para valor bruto da coluna1

    return pd.Series([chave_final(i) for i in range(len(df))], index=df.index)

### Gera a Chave de Material

In [53]:
def formatar_material(coluna):
    def extrair_codigo(texto):
        #Tenta extrair o padrão: 2+ letras + 1+ números
        match = re.search(r'\b([A-Za-z]{2,}\s*\d{1,})\b', texto)
        if match:
            return match.group(1).upper().strip()
    return coluna.apply(extrair_codigo)

### Relatório de Vendas

In [54]:
def resumo_por_ano(df, ano):
    df_ano = df[df[ano].notnull() & (df[ano] > 0)]
    return (
        df_ano.groupby(['KeyManuf', 'Distributor'])
        .agg(
            **{f'Produtos_{ano}': ('Material', lambda x: len(set(i for i in x if i))),
               f'Volume_{ano}': (ano, 'sum'),
               f'Lista_{ano}': ('Material', lambda x: sorted(set(i for i in x if i)))}
        )
    )

In [55]:
# Filtra apenas linhas que contenham 'BELSIL' na descrição (case-insensitive)
df_sale = df_sale[df_sale['Material'].str.contains('BELSIL', case=False, na=False)]

# Limpa e padroniza a coluna 'Material'
df_sale['Material'] = formatar_material(df_sale['Material'].astype(str))

# Padroniza fabricantes e distribuidores
df_sale['KeyManuf'] = gerar_chaves(df_sale, 'Customer')

# Gera os resumos por ano
resumo_2023 = resumo_por_ano(df_sale, '2023')
resumo_2024 = resumo_por_ano(df_sale, '2024')
resumo_2025 = resumo_por_ano(df_sale, '2025')

# Junta os resultados de 2023 e 2024
df_final = pd.concat([resumo_2023, resumo_2024, resumo_2025], axis=1).reset_index()

# Funções para itens adicionados e perdidos
def itens_adicionados(row):
    l2023 = set(row['Lista_2023']) if isinstance(row['Lista_2023'], list) else set()
    l2024 = set(row['Lista_2024']) if isinstance(row['Lista_2024'], list) else set()
    l2025 = set(row['Lista_2025']) if isinstance(row['Lista_2025'], list) else set()
    return sorted((l2024 | l2025) - l2023)

def itens_perdidos(row):
    l2023 = set(row['Lista_2023']) if isinstance(row['Lista_2023'], list) else set()
    l2024 = set(row['Lista_2024']) if isinstance(row['Lista_2024'], list) else set()
    return sorted(l2023 - l2024)

# Função para unir todos os códigos vendidos (2023 e 2024)
def Itens_Vendidos(row):
    l1 = set(row['Lista_2023']) if isinstance(row['Lista_2023'], list) else set()
    l2 = set(row['Lista_2024']) if isinstance(row['Lista_2024'], list) else set()
    l3 = set(row['Lista_2025']) if isinstance(row['Lista_2025'], list) else set()
    return sorted(l1 | l2 | l3)

# Cria as colunas de interesse
df_final['Itens_Adicionados_2024_2025'] = df_final.apply(itens_adicionados, axis=1)
df_final['Itens_Perdidos_2024'] = df_final.apply(itens_perdidos, axis=1)
df_final['Itens_Vendidos'] = df_final.apply(Itens_Vendidos, axis=1)

# Seleciona as colunas finais
df_final = df_final[
    ['KeyManuf', 'Distributor',
     'Produtos_2023', 'Produtos_2024', 'Produtos_2025',
     'Volume_2023', 'Volume_2024', 'Volume_2025',
     'Itens_Adicionados_2024_2025', 'Itens_Perdidos_2024',
     'Itens_Vendidos']
]

# Salva o resultado em Excel
df_final.to_excel('Vendas_BELSIL_por_ano.xlsx', index=False)
df_final.head()

,KeyManuf,Distributor,Produtos_2023,Produtos_2024,Produtos_2025,Volume_2023,Volume_2024,Volume_2025,Itens_Adicionados_2024_2025,Itens_Perdidos_2024,Itens_Vendidos
0,2k,EMBACAPS,4.0,4.0,3.0,0.3020,0.13015,0.1235,"[CM 740, DM 350]",[DM 6010],"[CM 040, CM 740, DM 0, DM 1000, DM 350, DM 6010]"
1,3x quimicos,BRENNTAG QUIMICA BRASIL,1.0,NaN,NaN,0.1900,NaN,NaN,[],[DM 1000],[DM 1000]
2,a &,FOCUS QUIMICA,9.0,5.0,6.0,8.5000,1.50800,1.8670,"[ADM 8301, EG 5, TMS 803]","[CM 1000, DM 350, DM 6010, GB 150, OW 2100]","[ADM 8301, ADM 9000, CM 1000, CM 740, DM 350, ..."
3,a g,EMBACAPS,3.0,NaN,NaN,0.0410,NaN,NaN,[],"[CM 1000, DM 350, EG 5]","[CM 1000, DM 350, EG 5]"
4,a jose,MORAIS DE CASTRO,1.0,NaN,NaN,0.0475,NaN,NaN,[],[DM 1000],[DM 1000]


# Análise Relatórios Lançamentos

## Descritivo Lançamentos





In [56]:
#Carrega o banco de dados como tabela
data = '/content/'+filename
df_launches = pd.read_excel(data)
pd.options.display.max_columns = None
df_launches.nunique()

,0
Número do Produto,4757
Data de Publicação,246
Produto,3123
Marca,2819
Empresa,899
Categoria,3
Sub-Categoria,33
Descrição do Produto,4752
Preço por 100g/ml,3196
Preço em moeda local,1574


In [57]:
#Edita Coluna Ano
df_launches['Ano'] = pd.to_datetime(df_launches['Data de Publicação']).dt.year

##Upload INCI

In [58]:
data = '/content/'+filename2
df_inci = pd.read_excel(data)
pd.options.display.max_columns=None
df_inci.nunique()

,0
Produto,51
Ingrediente,50
Prioridade,3


## Função Analisa Ingredientes

In [59]:
#this checks if any combination of INCI as present in Ingredient
def verifica_ingrediente(formula,produto,material):
  quantidade = len(produto.difference(formula))
  if quantidade == 0:
    return material
  return None

In [60]:
#If last code is true, this returns the Descrição name
def procura_produtos(formula, produtos, materiais):
  formula = formula.copy()
  resultados = []
  for prod, mat in zip(produtos, materiais):
    resultado = verifica_ingrediente(formula, prod, mat)
    if resultado is not None:
      formula = formula.difference(prod) ## Para remover os ingredientes já encontrados numa nova busca.
      resultados.append(resultado)
  return resultados

In [61]:
# Função para sinalizar ingredientes do dictOTHERS
def sinaliza_ingredientes(x):
    ingredientes = set().union(*x)  # une todos os sets/listas de ingredientes do grupo
    encontrados = set()
    for ing in ingredientes:
        for palavra in dictOTHERS:
            if palavra.lower() in ing.lower():
                encontrados.add(ing)
    return ', '.join(sorted(encontrados))

## Função Transpoe coluna

In [62]:
#this code transpose data. Used when we bring each category and the number of lauches.
def transpor (df, coluna, linha):
  for cat in df[coluna].unique():
    f = df[coluna] == cat
    df[cat] = df[f][linha]
    f = df[cat].isna()
    df.loc[f, cat] = df.loc[f, cat].apply(lambda x:[])

In [63]:
def to_set(x):
    s = set()
    for item in x:
        if isinstance(item, list):
            s.update(item)
        elif isinstance(item, str):
            s.add(item)
    return sorted(s)

In [64]:
def transpor_2 (df, coluna, linha):
  for cat in df[coluna].unique():
    f = df[coluna] == cat
    df[cat] = df[f][linha]
    f = df[cat].isna()
    df.loc[f, cat] = df.loc[f, cat].apply(lambda x:x)

## Contém Silicone?

In [65]:
#Dicionário Silicones geral
dictOTHERS = {'methicone':'1', 'Dimethicone':'1','methicone Crosspolymer':'1','methiconol':'1',
              'ylsiloxysilicate':'1', 'ylsilsesquioxane':'1', 'Disiloxane':'1', 'Silica':'1','siloxane':'1'}

In [66]:
df_launches['Silicone'] = df_launches['Ingredients (Standard form)'].str.extract('('+'|'.join(dictOTHERS)+')',expand=False).map(dictOTHERS)

## Gera chave de Material


In [67]:
#Edita tabela INCI

df_inci.dropna(inplace=True)
df_inci['Produto'] = formatar_material(df_inci['Produto'])

#Cria uma lista iterável dos ingredientes nos Produtos
df_inci['Ing'] = df_inci['Ingrediente'].str.split(', ').apply(set)
df_inci.sort_values('Prioridade', ascending=True, inplace=True)

In [68]:
#Dicionário de palavras a remover da coluna Ingredientes no Mintel
dictIng = {
    r'\s*and/or\s*': ',',
    r',\s*': ',',
    r'\s*,': ',',
    r'\s*\(and\)\s*': ','
}

In [69]:
# Cria coluna com produtos identificados
df_launches['Ingredients (Standard form)'] = df_launches['Ingredients (Standard form)'].astype(str)
#Cria uma lista iterável das ingredientes cosmeticos
df_launches['Ing'] = (
    df_launches['Ingredients (Standard form)']
    .replace(dictIng, regex=True)
    .str.split(',')
    .apply(set)
)

# Relaciona os ingredientes cosméticos
df_launches['Lançamentos'] = df_launches['Ing'].apply(
    lambda formulacao: procura_produtos(formulacao, df_inci['Ing'], df_inci['Produto'])
)

## Gera Chave Fabricante



In [70]:
df_launches['Fabricante']= df_launches['Fabricante'].astype(str)
df_launches['Marca']= df_launches['Marca'].astype(str)
df_launches['Empresa']= df_launches['Empresa'].astype(str)
df_launches['KeyManuf'] = gerar_chaves(df_launches, 'Fabricante', 'Empresa', 'Marca')
df_launches['KeyMarca'] = gerar_chaves(df_launches, 'Marca')
df_launches['KeyEmpresa'] = gerar_chaves(df_launches, 'Empresa')

## Relatório de Lançamentos

In [71]:
df_launches.head()

,Número do Produto,Data de Publicação,Produto,Marca,Empresa,Categoria,Sub-Categoria,Descrição do Produto,Preço por 100g/ml,Preço em moeda local,Tipo de Lançamento,Posicionamento,Ingredients (Standard form),Fabricante,Território da empresa fabricante,Local de fabricação,Manufacturer Company Address,Link da Imagem Primária,Ano,Silicone,Ing,Lançamentos,KeyManuf,KeyMarca,KeyEmpresa
0,13517914,2025-12-18,Shampoo,Tio Nacho Reconstrutor Total,Athenas Indústria e Tercerização de Cosméticos,Produtos para Cabelos,Xampu,Tio Nacho Reconstrutor Total (Total Reconstruc...,10.72,44.49,Relançamento,"Orgânico, Botânico/Herbóreo, Cabelos Danificad...","Aqua, Sodium Laureth Sulfate, Cocamidopropyl B...",Athenas Indústria e Tercerização de Cosméticos,Brazil,Brasil,Caieiras,https://media.mintel.com/i01/mediaserver/perfo...,2025,NaN,"{Citric Acid, Hexyl Cinnamal, Wheat Amino Acid...",[],athenas e,tio nacho,athenas e
1,13517916,2025-12-18,Leave-In Treatment,Tío Nacho Antiqueda Reconstrutor Total,Genomma Laboratories do Brasil,Produtos para Cabelos,Tratamento para Cabelos,Tío Nacho Antiqueda Reconstrutor Total (Anti F...,35.99,35.99,Nova Embalagem,"Orgânico, Botânico/Herbóreo, Cabelos Danificad...","Aqua, Cetyl Alcohol, Behentrimonium Chloride, ...",Athenas Indústria e Tercerização de Cosméticos,Brazil,Brasil,Caieiras,https://media.mintel.com/i01/mediaserver/perfo...,2025,1,"{Hexyl Cinnamal, Cetearyl Alcohol, Wheat Amino...",[GB 1020],athenas e,tio nacho,genomma laboratories
2,13517918,2025-12-18,2% Nanoencapsulated Retinol Renew Facial Serum,N°21 Ativ,Profarma Distribuidora de Produtos Farmaceuticos,Produtos para Pele,Cuidado Facial/Pescoço,N°21 Ativ Sérum Facial Retinol 2% Nanoencapsul...,166.63,49.99,Novo Produto,"Fortificado com Vitaminas/Minerais, Hipoalergê...","Aqua, Propanediol, Polysorbate 20, Glycerin, P...",Laboratório Industrial Farmacêutico - Lifar,Brazil,Brasil,Porto Alegre (RS),https://media.mintel.com/i01/mediaserver/perfo...,2025,NaN,"{BHT, Caprylyl Glycol, Retinol, Aqua, Xanthan ...",[],industrial farmaceutico,ndeg21 ativ,profarma farmaceuticos
3,13517920,2025-12-18,Aqua Protect Face Fluid Sunscreen SPF 70,Nº 21 Solar,Instituto Pasteur de Cosmiatria,Produtos para Pele,Sol - Exposição Solar/Câmara de Bronzeamento,Nº 21 Solar Aqua Protect Protetor Solar Fluído...,162.48,64.99,Novo Produto,"Proteção UV, Não Comedogênico, Testado Dermato...","Aqua, Octocrylene, Homosalate, Cyclopentasilox...",Instituto Pasteur de Cosmiatria,Brazil,Brasil,Porto Alegre - RS,https://media.mintel.com/i01/mediaserver/perfo...,2025,1,"{Butyl Methoxydibenzoylmethane, Caprylyl Glyco...",[],instituto pasteur,no solar,instituto pasteur
4,13517922,2025-12-18,Oiliness Control Deep Cleansing Gel,N°21,Profarma Distribuidora de Produtos Farmaceuticos,Produtos para Pele,Rosto - Limpadores,N°21 Gel de Limpeza Profunda Controle de Oleos...,16.66,49.99,Nova Variedade/Extensão de Linha,"Hipoalergênico, Testado Dermatologicamente, Ex...","Aqua, Sodium Laureth Sulfate, Propylene Glycol...",Laboratório Industrial Farmacêutico - Lifar,Brazil,Brasil,Porto Alegre (RS),https://media.mintel.com/i01/mediaserver/perfo...,2025,NaN,"{Citric Acid, Menthol, Zinc PCA, Benzoic Acid,...",[],industrial farmaceutico,ndeg21,profarma farmaceuticos


In [72]:
df_launches['indice']=1
transpor_2(df_launches, 'Categoria', 'indice')

In [73]:
#renomeia coluna categoria
df_launches.rename(columns={'Produtos para Pele':'Pele', 'Produtos para Cabelos':'Cabelos',
                            'Maquilagem': 'Make',
                            'Número do Produto':'Total Lançamentos'},inplace=True)

In [74]:
df_c = df_launches.groupby(['KeyManuf'], dropna=False)[['Pele', 'Cabelos', 'Make']].apply(lambda x:x.count())

In [75]:
#cria tabela que lista o fabricante, o estado e o total de lançamentos, quais tem silicone,
df_b = df_launches.groupby(['KeyManuf'], dropna=False).agg({
    'KeyMarca': to_set,
    'KeyEmpresa': to_set,
    'Total Lançamentos': 'count',
    'Silicone': 'count',
    'Lançamentos': to_set,
    'Ing': sinaliza_ingredientes
})

In [76]:
#Une tabelas anteriores
df_launches_fab = pd.concat([df_c, df_b], axis=1).reset_index()

In [77]:
df_launches_fab.to_excel('Relatório Lançamentos.xlsx')
df_launches_fab.head()

,KeyManuf,Pele,Cabelos,Make,KeyMarca,KeyEmpresa,Total Lançamentos,Silicone,Lançamentos,Ing
0,&co e,6,1,3,"[beyoung, ollie, pink cheeks, pinkcheeks, pink...","[&co e, ollie]",10,8,"[ES 3007, GB 150, TMS 803]","C30-45 Alkyl Methicone, Cyclomethicone, Cyclop..."
1,3fa,0,4,0,"[we pink, weblue]",[3fa],4,4,[GB 1020],"Cyclopentasiloxane, Dimethicone, Dimethiconol"
2,5s e,1,0,0,[fenzza make],[5s e],1,0,[],
3,a &,1,0,0,[oroskin],[a &],1,1,[OW 2100],PEG-12 Dimethicone
4,a tech,0,1,0,[kerasys hair],[a tech],1,1,"[ADM 1320, GB 150]","Amodimethicone, Amodimethicone/Morpholinomethy..."


## Agrupar relatório Vendas e Projetos

In [78]:
#agrupa lançamentos com vendas e projetos
df_launches_vend_proj = df_launches_fab.merge(df_final, on='KeyManuf', how='outer')

In [79]:
df_launches_vend_proj['Tem_Lancamento'] = ~df_launches_vend_proj['Total Lançamentos'].isna()
df_launches_vend_proj['Tem_Venda'] = (
    ~df_launches_vend_proj['Volume_2024'].isna() &
    ~df_launches_vend_proj['Volume_2025'].isna()
)

In [80]:
df_launches_vend_proj.head()

,KeyManuf,Pele,Cabelos,Make,KeyMarca,KeyEmpresa,Total Lançamentos,Silicone,Lançamentos,Ing,Distributor,Produtos_2023,Produtos_2024,Produtos_2025,Volume_2023,Volume_2024,Volume_2025,Itens_Adicionados_2024_2025,Itens_Perdidos_2024,Itens_Vendidos,Tem_Lancamento,Tem_Venda
0,&co e,6.0,1.0,3.0,"[beyoung, ollie, pink cheeks, pinkcheeks, pink...","[&co e, ollie]",10.0,8.0,"[ES 3007, GB 150, TMS 803]","C30-45 Alkyl Methicone, Cyclomethicone, Cyclop...",FOCUS QUIMICA,NaN,3.0,3.0,NaN,1.00200,0.3980,"[DM 5, EG 5, EG 6000, TMS 803]",[],"[DM 5, EG 5, EG 6000, TMS 803]",True,True
1,2k,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,EMBACAPS,4.0,4.0,3.0,0.302,0.13015,0.1235,"[CM 740, DM 350]",[DM 6010],"[CM 040, CM 740, DM 0, DM 1000, DM 350, DM 6010]",False,True
2,2k,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,FOCUS QUIMICA,NaN,1.0,NaN,NaN,0.01500,NaN,[GB 1020],[],[GB 1020],False,False
3,3fa,0.0,4.0,0.0,"[we pink, weblue]",[3fa],4.0,4.0,[GB 1020],"Cyclopentasiloxane, Dimethicone, Dimethiconol",FOCUS QUIMICA,NaN,1.0,3.0,NaN,0.03000,0.3760,"[DM 0, EG 5, GB 1020, OW 2100]",[],"[DM 0, EG 5, GB 1020, OW 2100]",True,True
4,3x quimicos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BRENNTAG QUIMICA BRASIL,1.0,NaN,NaN,0.190,NaN,NaN,[],[DM 1000],[DM 1000],False,False


# Generate a Report will all information

In [81]:
df_launches_vend_proj.to_excel('Final.xlsx')